# Prepare PDF Documents

This notebook corresponds to `src/ingestion/prepare_pdf_documents.py`.

Purpose:

```text
Data/raw/pdfs/  →  Data/documents.json
```

It recursively reads all PDF files under `Data/raw/pdfs/`, extracts text using `pypdf`, and stores each PDF as one document with `doc_id`, `title`, `source`, `content`, and metadata.

In [ ]:
from pathlib import Path
import json
from pypdf import PdfReader
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PDF_ROOT_DIR = PROJECT_ROOT / "Data" / "raw" / "pdfs"
OUTPUT_PATH = PROJECT_ROOT / "Data" / "documents.json"

print("Project root:", PROJECT_ROOT)
print("PDF root exists:", PDF_ROOT_DIR.exists())
print("Output path:", OUTPUT_PATH)

In [ ]:
pdf_files = sorted(PDF_ROOT_DIR.rglob("*.pdf"))
print("Number of PDF files:", len(pdf_files))
for p in pdf_files[:10]:
    print(p.relative_to(PDF_ROOT_DIR))

In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> tuple[str, int]:
    reader = PdfReader(str(pdf_path))
    pages_text = []

    for page in reader.pages:
        text = page.extract_text() or ""
        text = text.strip()

        if text:
            pages_text.append(text)

    return "\n\n".join(pages_text), len(reader.pages)


def clean_doc_id(text: str) -> str:
    return (
        text.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace(",", "")
        .replace("__", "_")
    )

In [ ]:
def prepare_pdf_documents(
    pdf_root_dir: Path,
    output_path: Path,
    min_length: int = 300,
) -> list[dict]:
    documents = []

    pdf_files = sorted(pdf_root_dir.rglob("*.pdf"))

    for pdf_path in pdf_files:
        print(f"Processing {pdf_path.relative_to(pdf_root_dir)}...")

        content, page_count = extract_text_from_pdf(pdf_path)

        if len(content) < min_length:
            print(f"Skipped {pdf_path.name}: extracted text too short.")
            continue

        category = pdf_path.parent.name
        file_stem = pdf_path.stem

        doc_id = clean_doc_id(f"{category}_{file_stem}")
        title = f"{category} - {file_stem}"

        document = {
            "doc_id": doc_id,
            "title": title,
            "source": str(pdf_path),
            "source_path": str(pdf_path),
            "source_html": str(pdf_path),  # keep this for compatibility with existing code
            "content": content,
            "metadata": {
                "file_type": "pdf",
                "category": category,
                "page_count": page_count,
                "original_filename": pdf_path.name,
                "relative_path": str(pdf_path.relative_to(pdf_root_dir)),
            },
        }

        documents.append(document)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)

    return documents

In [ ]:
documents = prepare_pdf_documents(
    pdf_root_dir=PDF_ROOT_DIR,
    output_path=OUTPUT_PATH,
)

print(f"Saved {len(documents)} documents to {OUTPUT_PATH}")

In [ ]:
preview = pd.DataFrame([
    {
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "category": doc["metadata"].get("category"),
        "page_count": doc["metadata"].get("page_count"),
        "content_length": len(doc["content"]),
    }
    for doc in documents
])

preview.head(20)

In [ ]:
preview[["page_count", "content_length"]].describe()

In [ ]:
if documents:
    print("Sample document ID:", documents[0]["doc_id"])
    print("Title:", documents[0]["title"])
    print("Content preview:")
    print(documents[0]["content"][:1500])